# GC 1-Minute – Barrier Strategy with Gradient Boosting

Instead of forecasting the price, this notebook learns the trading rule directly. For every 1-minute candle and every setup:

> **Long:** is the price `+target` ticks above the entry (close of the candle) **before** it is `stop` ticks below,
> within `horizon` minutes? – **Short:** the same mirrored.

A target reached in minute 13 of a 15-minute window counts as a win. Two classifiers per setup (long, short) estimate these
probabilities; a trade is taken only when a probability is above a threshold that is chosen on the **validation period**
and then applied **once** to the separate test CSV.

**Inputs:** only the CSV columns Open, Close, High, Low, Ask, Bid, Delta, Delta/Volume, Session Delta, Session Delta %,
Max Delta, Min Delta, Delta Change, Volume – plus the time of day / weekday from the timestamp (`USE_TIME_FEATURES`).
All other CSV columns are ignored. The timestamp and all original columns stay in the output CSV.

**Time split without overlap:** training data = everything in the training CSV **before** the first candle of the test CSV.
Of that, the last `VALIDATION_FRACTION` is validation (threshold selection); the test CSV is only used at the very end.

**Comparison:** on the test data the model is compared with *random direction* and *always long* using the same entry
times and the same exit rules. The model is only useful if it is clearly better than both **after costs**.

## 0. Configuration

In [1]:
from pathlib import Path

SYMBOL = "GC"
TRAIN_CSV_PATH = Path(r"E:\Programierung\Daten\Candelexport_GC_M1_25_11_2018_20_10_2026.csv")
TEST_CSV_PATH = Path(r"E:\Programierung\Daten\Test_GC_M1_20_10_2025_20_10_2026.csv")
TEST_OUTPUT_CSV_PATH = TEST_CSV_PATH.with_name(TEST_CSV_PATH.stem + "_barrier_signals.csv")
MODEL_DIR = Path("models_barrier")
LOG_PATH = MODEL_DIR / "barrier_log.csv"

# Model inputs: ONLY these CSV columns (+ time features from "Time UTC" if USE_TIME_FEATURES)
INPUT_COLUMNS = [
    "Open", "Close", "High", "Low", "Ask", "Bid", "Delta", "Delta/Volume",
    "Session Delta", "Session Delta %", "Max Delta", "Min Delta", "Delta Change", "Volume",
]
PERCENT_COLUMNS = ["Delta/Volume", "Session Delta %"]   # exported as text "59.15%" -> 0.5915
USE_TIME_FEATURES = True                                 # hour of day + weekday from the timestamp

# Additional indicators (calculated only from the input columns above)
RSI_PERIODS = [7, 14, 21, 50]   # RSI (Wilder) on 1-minute closes
TWAP_WINDOWS = [15, 60]         # rolling TWAP in minutes (+ session TWAP, reset after every pause / session start)
VWAP_WINDOWS = [15, 60]         # rolling VWAP in minutes (+ session VWAP, reset after every pause / session start)

# Trade setups (horizon in minutes). Two kinds of barriers:
#   mode "vol":   target/stop = sigma x expected move over the horizon
#                 (std of 1-minute close changes over the last VOL_LOOKBACK candles x sqrt(horizon)), in ticks
#                 -> a trade means the same in quiet and in wild markets
#   mode "fixed": target_ticks / stop_ticks as fixed numbers, e.g.
#                 {"name": "15min_fix", "horizon": 15, "mode": "fixed", "target_ticks": 30, "stop_ticks": 30}
SETUPS = [
    {"name": "15min", "horizon": 15, "mode": "vol", "target_sigma": 1.0, "stop_sigma": 1.0},
    {"name": "60min", "horizon": 60, "mode": "vol", "target_sigma": 1.0, "stop_sigma": 1.0},
]
VOL_LOOKBACK = 60               # candles for the volatility estimate (only past data)
MIN_BARRIER_TICKS = 5           # volatility barriers are never smaller than this

# Futures contract and execution
TICK_SIZE = 0.10                # GC / MGC
TICK_VALUE_USD = 10.0           # GC: 10 USD per tick and contract (MGC: 1 USD)
COST_PER_TRADE_TICKS = 2.0      # round trip incl. spread/slippage and commission -> adjust to your account
TARGET_FILL_EXTRA_TICKS = 1     # target counts only if the price trades this many ticks beyond it
MAX_GAP_MINUTES = 5             # no trade if the window contains a longer pause (session break, weekend)
WARMUP_CANDLES = 60             # first candles of a file are not traded (rolling inputs not complete yet)

# Data split and training
VALIDATION_FRACTION = 0.2       # last 20 % of the training data (before the test period) -> threshold selection
TRAIN_ROW_STRIDE = 1            # 1 = every candle; e.g. 3 = every 3rd candle (faster, slightly less data)
GBM_PARAMS = dict(
    max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=500,
    l2_regularization=1.0, random_state=42,
)

# Trading rule: trade if max(P_long, P_short) >= threshold; the threshold is chosen on validation (best t-value)
PROB_THRESHOLDS = [round(0.30 + 0.025 * i, 3) for i in range(21)]   # 0.30 ... 0.80
MIN_VALIDATION_TRADES = 100     # thresholds with fewer validation trades are not eligible
BASELINE_RANDOM_RUNS = 5
COMPUTE_FEATURE_IMPORTANCE = True   # permutation importance on a validation sample (a few minutes)

# Direction strategy (section 8.2): direction from the direction model, timing from the win models.
# All combinations are evaluated on validation; the one with the best t-value is applied ONCE to the test data.
DIRECTION_QUANTILES = [0.05, 0.10, 0.20]   # trade the lowest / highest X % of P(up first) -> short / long
TIMING_QUANTILES = [0.0, 0.5, 0.75]        # also require P(win) of the traded side >= this validation quantile (0 = off)
DIRECTION_SIDES = ["both", "short", "long"]

MODEL_DIR.mkdir(parents=True, exist_ok=True)
for p in (TRAIN_CSV_PATH, TEST_CSV_PATH):
    if not p.is_file():
        raise FileNotFoundError(p)
print(f"Train CSV: {TRAIN_CSV_PATH}\nTest CSV:  {TEST_CSV_PATH}")
for s in SETUPS:
    if s.get("mode", "fixed") == "vol":
        print(f"Setup {s['name']}: target {s['target_sigma']} σ / stop {s['stop_sigma']} σ of the expected "
              f"{s['horizon']}-minute move (ticks are shown after the labels are computed)")
    else:
        be = (s["stop_ticks"] + COST_PER_TRADE_TICKS) / (s["target_ticks"] + s["stop_ticks"])
        print(f"Setup {s['name']}: +{s['target_ticks']} / -{s['stop_ticks']} ticks within {s['horizon']} min "
              f"-> break-even win rate ≈ {100 * be:.1f} % (ignoring time exits)")

Train CSV: E:\Programierung\Daten\Candelexport_GC_M1_25_11_2018_20_10_2026.csv
Test CSV:  E:\Programierung\Daten\Test_GC_M1_20_10_2025_20_10_2026.csv
Setup 15min: target 1.0 σ / stop 1.0 σ of the expected 15-minute move (ticks are shown after the labels are computed)
Setup 60min: target 1.0 σ / stop 1.0 σ of the expected 60-minute move (ticks are shown after the labels are computed)


## 1. Imports

In [2]:
import json
import time
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

try:
    from IPython.display import display
except ImportError:  # outside Jupyter
    display = print

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)

## 2. Load CSV

In [3]:
def load_csv(csv_path: Path) -> pd.DataFrame:
    """Load an ATAS 1-minute export in Bar order; only the input columns, the timestamp and Bar are kept."""
    raw = pd.read_csv(csv_path, sep=";", encoding="utf-8-sig")
    missing = [c for c in ["Bar", "Time UTC"] + INPUT_COLUMNS if c not in raw.columns]
    if missing:
        raise ValueError(f"{csv_path.name}: missing columns {missing}")

    for col in PERCENT_COLUMNS:
        text = (raw[col].astype(str).str.strip().str.replace("%", "", regex=False)
                .str.replace("'", "", regex=False).str.replace(" ", "", regex=False))
        text = text.where(text.str.contains(".", regex=False), text.str.replace(",", ".", regex=False))
        values = pd.to_numeric(text, errors="coerce")
        n_bad = int((values.isna() & (text != "") & (text.str.lower() != "nan")).sum())
        if n_bad:
            print(f"⚠ {col}: {n_bad:,} values not parseable -> 0")
        raw[col] = (values / 100).fillna(0.0)

    df = raw[["Bar", "Time UTC"] + INPUT_COLUMNS].copy()
    for col in INPUT_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
    df["timestamp"] = pd.to_datetime(df["Time UTC"], format="%d.%m.%Y %H:%M:%S", utc=True)
    if not df["Bar"].is_monotonic_increasing:
        df = df.sort_values("Bar", kind="stable")
    df = df.drop(columns=["Time UTC"]).reset_index(drop=True)
    if df[["Open", "High", "Low", "Close"]].isna().any().any():
        raise ValueError(f"{csv_path.name}: missing OHLC values")
    print(f"{csv_path.name}: {len(df):,} candles, {df['timestamp'].min()} → {df['timestamp'].max()}")
    return df


df_all = load_csv(TRAIN_CSV_PATH)
df_test = load_csv(TEST_CSV_PATH)

# Training data = everything before the first candle of the test CSV (no overlap)
test_start = df_test["timestamp"].min()
df_train = df_all[df_all["timestamp"] < test_start].reset_index(drop=True)
del df_all
print(f"Training data used: {len(df_train):,} candles until {df_train['timestamp'].max()} (test starts {test_start})")

Candelexport_GC_M1_25_11_2018_20_10_2026.csv: 2,796,096 candles, 2018-11-25 23:00:00+00:00 → 2026-10-20 20:59:00+00:00
Test_GC_M1_20_10_2025_20_10_2026.csv: 354,649 candles, 2025-10-20 00:00:00+00:00 → 2026-10-20 20:59:00+00:00
Training data used: 2,441,447 candles until 2025-10-19 23:59:00+00:00 (test starts 2025-10-20 00:00:00+00:00)


## 3. Inputs (features)

Only backward-looking values of the input columns: candle shape, price changes, volatility and position in the recent range
(all in ticks, independent of the price level), plus order flow (Ask/Bid, Delta, Session Delta, Max/Min Delta, Volume)
as raw values, ratios and rolling sums. Gradient boosting handles missing values itself.

**Indicators:** RSI for every period in `RSI_PERIODS` (level + change over 5 minutes), **TWAP** and **VWAP** – each as
session value (reset at every session start / after a pause) and as rolling value over `TWAP_WINDOWS` / `VWAP_WINDOWS`.
TWAP/VWAP enter as **distance of the close in ticks** (and relative to the recent candle range), so the model does not
see absolute price levels. Typical price = (High + Low + Close) / 3.

In [4]:
def rsi(close: pd.Series, period: int) -> pd.Series:
    """Relative Strength Index with Wilder smoothing (0..100)."""
    diff = close.diff()
    avg_gain = diff.clip(lower=0).ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = (-diff.clip(upper=0)).ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    value = 100 - 100 / (1 + avg_gain / avg_loss.replace(0, np.nan))
    value[(avg_loss == 0) & (avg_gain > 0)] = 100.0
    value[(avg_loss == 0) & (avg_gain == 0)] = 50.0
    return value


def session_ids(ts: pd.Series) -> pd.Series:
    """New session after every pause > MAX_GAP_MINUTES (daily break, weekend)."""
    return (ts.diff().dt.total_seconds().div(60).fillna(np.inf) > MAX_GAP_MINUTES).cumsum()


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    o, h, l, c = df["Open"], df["High"], df["Low"], df["Close"]
    vol, ask, bid, delta = df["Volume"], df["Ask"], df["Bid"], df["Delta"]
    t = TICK_SIZE
    f = {}

    # Candle shape and price changes (ticks)
    f["body_ticks"] = (c - o) / t
    f["range_ticks"] = (h - l) / t
    f["upper_wick_ticks"] = (h - np.maximum(o, c)) / t
    f["lower_wick_ticks"] = (np.minimum(o, c) - l) / t
    ret1 = (c - c.shift(1)) / t
    for k in [1, 3, 5, 10, 15, 30, 60]:
        f[f"ret_{k}_ticks"] = (c - c.shift(k)) / t
    for k in [5, 15, 60]:
        hh, ll = h.rolling(k).max(), l.rolling(k).min()
        f[f"range_mean_{k}"] = f["range_ticks"].rolling(k).mean()
        f[f"ret_std_{k}"] = ret1.rolling(k).std()
        f[f"pos_in_range_{k}"] = (c - ll) / (hh - ll).replace(0, np.nan)
        f[f"dist_high_{k}_ticks"] = (hh - c) / t
        f[f"dist_low_{k}_ticks"] = (c - ll) / t

    # Order flow
    f["volume"] = vol
    f["log_volume"] = np.log1p(vol.clip(lower=0))
    f["ask"] = ask
    f["bid"] = bid
    f["ask_bid_imbalance"] = (ask - bid) / (ask + bid).replace(0, np.nan)
    f["delta"] = delta
    f["delta_per_volume"] = df["Delta/Volume"]
    f["session_delta"] = df["Session Delta"]
    f["session_delta_pct"] = df["Session Delta %"]
    f["session_delta_change_15"] = df["Session Delta"] - df["Session Delta"].shift(15)
    f["max_delta"] = df["Max Delta"]
    f["min_delta"] = df["Min Delta"]
    f["delta_change"] = df["Delta Change"]
    f["max_delta_rel"] = df["Max Delta"] / vol.replace(0, np.nan)
    f["min_delta_rel"] = df["Min Delta"] / vol.replace(0, np.nan)
    for k in [1, 2, 3, 5]:
        f[f"delta_lag_{k}"] = delta.shift(k)
        f[f"delta_per_volume_lag_{k}"] = df["Delta/Volume"].shift(k)
    for k in [5, 15, 60]:
        f[f"delta_sum_{k}"] = delta.rolling(k).sum()
        f[f"delta_ratio_{k}"] = delta.rolling(k).sum() / vol.rolling(k).sum().replace(0, np.nan)
        f[f"volume_rel_{k}"] = vol / vol.rolling(k).mean().replace(0, np.nan)
        f[f"ask_bid_imbalance_{k}"] = ((ask - bid).rolling(k).sum()
                                       / (ask + bid).rolling(k).sum().replace(0, np.nan))

    # RSI
    for p in RSI_PERIODS:
        r = rsi(c, p)
        f[f"rsi_{p}"] = r
        f[f"rsi_{p}_change_5"] = r - r.shift(5)

    # TWAP / VWAP (distance of the close in ticks and relative to the recent candle range)
    tp = (h + l + c) / 3
    session = session_ids(df["timestamp"])
    range_60 = ((h - l) / t).rolling(60, min_periods=5).mean().replace(0, np.nan)
    twap_session = tp.groupby(session).cumsum() / (tp.groupby(session).cumcount() + 1)
    vol_cum = vol.groupby(session).cumsum()
    vwap_session = ((tp * vol).groupby(session).cumsum() / vol_cum.replace(0, np.nan)).groupby(session).ffill()
    averages = {"twap_session": twap_session, "vwap_session": vwap_session}
    for k in TWAP_WINDOWS:
        averages[f"twap_{k}"] = tp.rolling(k).mean()
    for k in VWAP_WINDOWS:
        averages[f"vwap_{k}"] = (tp * vol).rolling(k).sum() / vol.rolling(k).sum().replace(0, np.nan)
    for name, avg in averages.items():
        f[f"dist_{name}_ticks"] = (c - avg) / t
        f[f"dist_{name}_rel"] = (c - avg) / t / range_60
    f["vwap_minus_twap_session_ticks"] = (vwap_session - twap_session) / t

    # Time of day / weekday from the timestamp (UTC)
    if USE_TIME_FEATURES:
        ts = df["timestamp"]
        minute_of_day = ts.dt.hour * 60 + ts.dt.minute
        f["tod_sin"] = np.sin(2 * np.pi * minute_of_day / 1440)
        f["tod_cos"] = np.cos(2 * np.pi * minute_of_day / 1440)
        f["weekday"] = ts.dt.dayofweek
        f["minutes_since_gap"] = minutes_since_gap(ts)

    out = pd.DataFrame(f).replace([np.inf, -np.inf], np.nan).astype("float32")
    return out


def minutes_since_gap(ts: pd.Series) -> pd.Series:
    """Minutes since the last pause > MAX_GAP_MINUTES (session start / after the weekend)."""
    gap = ts.diff().dt.total_seconds().div(60).fillna(np.inf) > MAX_GAP_MINUTES
    block = gap.cumsum()
    return ts.groupby(block).cumcount().astype("float32")


t0 = time.time()
X_train_all = build_features(df_train)
X_test = build_features(df_test)
FEATURE_COLUMNS = list(X_train_all.columns)
print(f"{len(FEATURE_COLUMNS)} inputs built in {time.time() - t0:.0f} s")
print("Indicator inputs:", [c for c in FEATURE_COLUMNS if c.startswith(("rsi_", "dist_twap", "dist_vwap", "vwap_minus"))])

86 inputs built in 4 s
Indicator inputs: ['rsi_7', 'rsi_7_change_5', 'rsi_14', 'rsi_14_change_5', 'rsi_21', 'rsi_21_change_5', 'rsi_50', 'rsi_50_change_5', 'dist_twap_session_ticks', 'dist_twap_session_rel', 'dist_vwap_session_ticks', 'dist_vwap_session_rel', 'dist_twap_15_ticks', 'dist_twap_15_rel', 'dist_twap_60_ticks', 'dist_twap_60_rel', 'dist_vwap_15_ticks', 'dist_vwap_15_rel', 'dist_vwap_60_ticks', 'dist_vwap_60_rel', 'vwap_minus_twap_session_ticks']


## 4. Labels: Barrier Outcomes for Every Candle

For every candle, a long and a short trade entered at its close are played through minute by minute:
target (+`TARGET_FILL_EXTRA_TICKS` beyond it) → win, stop → loss (stop wins if both are touched in the same candle),
neither → closed at the close after `horizon` minutes. Windows that contain a pause are invalid (not traded, not trained).

With `mode: "vol"` target and stop are set per candle from the volatility of the **past** `VOL_LOOKBACK` candles.

**Direction label (for the pure direction test):** is the upper barrier (+target) touched **before** the lower one
(−target)? Only windows where exactly one side is reached first count; time-outs and candles touching both are excluded.
This removes the volatility question – what remains is only "up or down".

In [5]:
EXIT_TARGET, EXIT_STOP, EXIT_TIME = 1, 2, 3


def barrier_ticks(df: pd.DataFrame, setup: dict):
    """Target and stop in ticks per candle (arrays)."""
    n = len(df)
    if setup.get("mode", "fixed") == "fixed":
        return np.full(n, float(setup["target_ticks"])), np.full(n, float(setup["stop_ticks"]))
    ret1 = df["Close"].diff() / TICK_SIZE
    expected_move = (ret1.rolling(VOL_LOOKBACK, min_periods=VOL_LOOKBACK // 2).std() * np.sqrt(setup["horizon"])).to_numpy()
    target = np.maximum(np.round(setup["target_sigma"] * expected_move), MIN_BARRIER_TICKS)
    stop = np.maximum(np.round(setup["stop_sigma"] * expected_move), MIN_BARRIER_TICKS)
    return target, stop


def barrier_outcomes(df: pd.DataFrame, setup: dict) -> dict:
    horizon = setup["horizon"]
    target_ticks, stop_ticks = barrier_ticks(df, setup)
    c = df["Close"].to_numpy("float64")
    h = df["High"].to_numpy("float64")
    l = df["Low"].to_numpy("float64")
    n = len(df)
    extra = TARGET_FILL_EXTRA_TICKS * TICK_SIZE
    out = {"target_ticks": target_ticks, "stop_ticks": stop_ticks}
    for side, d in (("long", 1), ("short", -1)):
        target = c + d * (target_ticks * TICK_SIZE + extra)
        stop = c - d * stop_ticks * TICK_SIZE
        result = np.full(n, np.nan)
        exit_type = np.zeros(n, dtype="int8")
        minutes = np.zeros(n, dtype="int16")
        done = np.zeros(n, dtype=bool)
        for k in range(1, horizon + 1):
            hk = np.full(n, np.nan)
            lk = np.full(n, np.nan)
            hk[:n - k], lk[:n - k] = h[k:], l[k:]
            if d == 1:
                stop_hit, tgt_hit = lk <= stop, hk >= target
            else:
                stop_hit, tgt_hit = hk >= stop, lk <= target
            stop_hit &= ~done
            tgt_hit &= ~done & ~stop_hit
            result[stop_hit], exit_type[stop_hit], minutes[stop_hit] = -stop_ticks[stop_hit], EXIT_STOP, k
            result[tgt_hit], exit_type[tgt_hit], minutes[tgt_hit] = target_ticks[tgt_hit], EXIT_TARGET, k
            done |= stop_hit | tgt_hit
        ck = np.full(n, np.nan)
        ck[:n - horizon] = c[horizon:]
        open_ = ~done
        result[open_] = d * (ck[open_] - c[open_]) / TICK_SIZE
        exit_type[open_], minutes[open_] = EXIT_TIME, horizon
        out[side] = {"result": result, "exit": exit_type, "minutes": minutes, "win": exit_type == EXIT_TARGET}

    # Direction label: +target before -target (symmetric, no extra tick); both in one candle / time-out -> unresolved
    up_level, down_level = c + target_ticks * TICK_SIZE, c - target_ticks * TICK_SIZE
    up_first = np.zeros(n, dtype=bool)
    resolved = np.zeros(n, dtype=bool)
    decided = np.zeros(n, dtype=bool)
    for k in range(1, horizon + 1):
        hk = np.full(n, np.nan)
        lk = np.full(n, np.nan)
        hk[:n - k], lk[:n - k] = h[k:], l[k:]
        up, down = (hk >= up_level) & ~decided, (lk <= down_level) & ~decided
        only_up, only_down = up & ~down, down & ~up
        up_first |= only_up
        resolved |= only_up | only_down
        decided |= up | down
    out["direction"] = {"up_first": up_first, "resolved": resolved}

    # Valid = complete window inside the file, no pause inside the window, warm-up done
    gap = (df["timestamp"].diff().dt.total_seconds().div(60).fillna(0).to_numpy() > MAX_GAP_MINUTES).astype(int)
    cum = np.cumsum(gap)
    idx = np.arange(n)
    end = np.minimum(idx + horizon, n - 1)
    out["valid"] = ((idx + horizon < n) & ((cum[end] - cum[idx]) == 0) & (idx >= WARMUP_CANDLES)
                    & np.isfinite(target_ticks) & np.isfinite(stop_ticks))
    return out


t0 = time.time()
outcomes_train = {s["name"]: barrier_outcomes(df_train, s) for s in SETUPS}
outcomes_test = {s["name"]: barrier_outcomes(df_test, s) for s in SETUPS}
print(f"Barrier outcomes computed in {time.time() - t0:.0f} s")
for s in SETUPS:
    o = outcomes_train[s["name"]]
    v = o["valid"]
    print(f"[{s['name']}] valid candles {v.sum():,} | win rate if ALWAYS traded: "
          f"long {100 * o['long']['win'][v].mean():.1f} %, short {100 * o['short']['win'][v].mean():.1f} % "
          f"| Ø result long {np.nanmean(o['long']['result'][v]):+.2f}, short {np.nanmean(o['short']['result'][v]):+.2f} ticks (gross)")
    for label, oo in (("training data", o), ("test data", outcomes_test[s["name"]])):
        vv = oo["valid"]
        print(f"      {label}: Ø target {np.mean(oo['target_ticks'][vv]):.1f} / Ø stop {np.mean(oo['stop_ticks'][vv]):.1f} ticks, "
              f"direction resolved in {100 * oo['direction']['resolved'][vv].mean():.1f} % of the windows, "
              f"up first {100 * oo['direction']['up_first'][vv & oo['direction']['resolved']].mean():.1f} %")

Barrier outcomes computed in 4 s
in] valid candles 2,414,459 | win rate if ALWAYS traded: long 24.6 %, short 25.1 % | Ø result long -0.25, short -0.29 ticks (gross)
      training data: Ø target 18.0 / Ø stop 18.0 ticks, direction resolved in 56.2 % of the windows, up first 49.6 %
      test data: Ø target 71.6 / Ø stop 71.6 ticks, direction resolved in 55.5 % of the windows, up first 47.8 %
in] valid candles 2,333,728 | win rate if ALWAYS traded: long 26.6 %, short 27.4 % | Ø result long -0.21, short -0.37 ticks (gross)
      training data: Ø target 36.2 / Ø stop 36.2 ticks, direction resolved in 57.2 % of the windows, up first 49.3 %
      test data: Ø target 144.3 / Ø stop 144.3 ticks, direction resolved in 56.8 % of the windows, up first 46.8 %


## 5. Time Split: Training / Validation

In [6]:
n_train_total = len(df_train)
val_start = int(n_train_total * (1 - VALIDATION_FRACTION))
max_horizon = max(s["horizon"] for s in SETUPS)
# Purge: training labels must not look into the validation period
train_rows = np.arange(0, val_start - max_horizon, TRAIN_ROW_STRIDE)
val_rows = np.arange(val_start, n_train_total)
print(f"Training:   {df_train['timestamp'].iloc[train_rows[0]]} → {df_train['timestamp'].iloc[train_rows[-1]]} "
      f"({len(train_rows):,} candles, stride {TRAIN_ROW_STRIDE})")
print(f"Validation: {df_train['timestamp'].iloc[val_rows[0]]} → {df_train['timestamp'].iloc[val_rows[-1]]} ({len(val_rows):,} candles)")
print(f"Test:       {df_test['timestamp'].iloc[0]} → {df_test['timestamp'].iloc[-1]} ({len(df_test):,} candles)")

Training:   2018-11-25 23:00:00+00:00 → 2024-06-04 04:27:00+00:00 (1,953,097 candles, stride 1)
Validation: 2024-06-04 05:29:00+00:00 → 2025-10-19 23:59:00+00:00 (488,290 candles)
Test:       2025-10-20 00:00:00+00:00 → 2026-10-20 20:59:00+00:00 (354,649 candles)


## 6. Train Long and Short Classifier per Setup

In [7]:
models = {}
auc_rows = []
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    tr = train_rows[o_tr["valid"][train_rows]]
    va = val_rows[o_tr["valid"][val_rows]]
    te = np.flatnonzero(o_te["valid"])
    models[name] = {}
    for side in ("long", "short"):
        t0 = time.time()
        clf = HistGradientBoostingClassifier(**GBM_PARAMS)
        clf.fit(X_train_all.iloc[tr], o_tr[side]["win"][tr])
        p_val = clf.predict_proba(X_train_all.iloc[va])[:, 1]
        p_te = clf.predict_proba(X_test.iloc[te])[:, 1]
        models[name][side] = clf
        auc_rows.append({
            "Setup": name, "Side": side, "Train candles": len(tr),
            "Base win rate val %": 100 * o_tr[side]["win"][va].mean(),
            "AUC validation": roc_auc_score(o_tr[side]["win"][va], p_val),
            "AUC test": roc_auc_score(o_te[side]["win"][te], p_te),
            "Ø P val": p_val.mean(), "P val 99%": np.percentile(p_val, 99),
            "Fit seconds": time.time() - t0,
        })
        print(f"[{name} {side}] trained in {time.time() - t0:.0f} s – AUC validation {auc_rows[-1]['AUC validation']:.4f}, "
              f"test {auc_rows[-1]['AUC test']:.4f}")

print("\nAUC = how well the model ranks winners above losers: 0.50 = no skill, 0.55 = weak, 0.60+ = useful")
display(pd.DataFrame(auc_rows).round(4))

in long] trained in 29 s – AUC validation 0.6044, test 0.5913
in short] trained in 27 s – AUC validation 0.6139, test 0.5830
in long] trained in 24 s – AUC validation 0.6144, test 0.5875
in short] trained in 23 s – AUC validation 0.6210, test 0.5830

AUC = how well the model ranks winners above losers: 0.50 = no skill, 0.55 = weak, 0.60+ = useful


,Setup,Side,Train candles,Base win rate val %,AUC validation,AUC test,Ø P val,P val 99%,Fit seconds
0,15min,long,1931524,26.7251,0.6044,0.5913,0.2390,0.4235,28.8896
1,15min,short,1931524,26.9117,0.6139,0.5830,0.2516,0.4265,27.2446
2,60min,long,1867038,28.1934,0.6144,0.5875,0.2390,0.4537,24.0256
3,60min,short,1867038,27.9958,0.6210,0.5830,0.2650,0.4773,23.2312


### 6.1 Which Inputs Matter (permutation importance on a validation sample)

In [8]:
if COMPUTE_FEATURE_IMPORTANCE:
    rng = np.random.default_rng(0)
    for s in SETUPS:
        name = s["name"]
        va = val_rows[outcomes_train[name]["valid"][val_rows]]
        sample = rng.choice(va, size=min(50_000, len(va)), replace=False)
        for side in ("long", "short"):
            imp = permutation_importance(
                models[name][side], X_train_all.iloc[sample], outcomes_train[name][side]["win"][sample],
                scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=-1,
            )
            top = (pd.Series(imp.importances_mean, index=FEATURE_COLUMNS).sort_values(ascending=False).head(12))
            print(f"[{name} {side}] AUC loss when an input is shuffled (top 12):")
            print("   " + ", ".join(f"{k} {v:.4f}" for k, v in top.items()))

in long] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0802, range_mean_60 0.0185, volume_rel_60 0.0168, range_mean_15 0.0157, tod_cos 0.0122, minutes_since_gap 0.0120, tod_sin 0.0083, volume 0.0033, range_mean_5 0.0022, rsi_21 0.0012, ask 0.0010, dist_twap_session_ticks 0.0006
in short] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0775, range_mean_60 0.0276, range_mean_15 0.0266, minutes_since_gap 0.0203, volume_rel_60 0.0170, tod_cos 0.0144, tod_sin 0.0076, range_mean_5 0.0063, volume 0.0040, ret_std_15 0.0028, delta_sum_60 0.0019, dist_low_60_ticks 0.0009
in long] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0865, minutes_since_gap 0.0250, tod_cos 0.0215, volume_rel_60 0.0101, range_mean_60 0.0096, tod_sin 0.0048, range_mean_15 0.0046, volume 0.0037, weekday 0.0013, dist_high_60_ticks 0.0011, ask 0.0010, vwap_minus_twap_session_ticks 0.0009
in short] AUC loss when an input is shuffled (top 12):
   ret_std_60 0.0783, minutes_since_gap 0.

### 6.2 Pure Direction Test: "Up or Down First?"

One classifier per setup on the direction label only (volatility removed). **AUC 0.50 = no direction information**,
0.52–0.55 = weak, above 0.55 = interesting. The bucket table shows on the test data whether higher model probabilities
really go along with more "up first" outcomes – that is what a trading signal needs.

In [9]:
direction_models, direction_rows = {}, []
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    ok_tr = o_tr["valid"] & o_tr["direction"]["resolved"]
    ok_te = o_te["valid"] & o_te["direction"]["resolved"]
    tr = train_rows[ok_tr[train_rows]]
    va = val_rows[ok_tr[val_rows]]
    te = np.flatnonzero(ok_te)
    t0 = time.time()
    clf = HistGradientBoostingClassifier(**GBM_PARAMS)
    clf.fit(X_train_all.iloc[tr], o_tr["direction"]["up_first"][tr])
    direction_models[name] = clf
    p_va = clf.predict_proba(X_train_all.iloc[va])[:, 1]
    p_te = clf.predict_proba(X_test.iloc[te])[:, 1]
    y_va, y_te = o_tr["direction"]["up_first"][va], o_te["direction"]["up_first"][te]
    direction_rows.append({
        "Setup": name, "Train windows": len(tr), "Up first val %": 100 * y_va.mean(), "Up first test %": 100 * y_te.mean(),
        "AUC direction validation": roc_auc_score(y_va, p_va), "AUC direction test": roc_auc_score(y_te, p_te),
        "Fit seconds": time.time() - t0,
    })
    buckets = pd.qcut(p_te, 10, labels=False, duplicates="drop")
    table = (pd.DataFrame({"bucket": buckets, "p": p_te, "up": y_te})
             .groupby("bucket").agg(windows=("up", "size"), mean_p=("p", "mean"), actual_up_pct=("up", "mean")))
    table["actual_up_pct"] *= 100
    print(f"[{name}] test: model probability (10 buckets, low -> high) vs. actual share 'up first'")
    display(table.round(3))

print("AUC 0.50 = no direction information | 0.52–0.55 = weak | > 0.55 = interesting")
display(pd.DataFrame(direction_rows).round(4))

in] test: model probability (10 buckets, low -> high) vs. actual share 'up first'


,windows,mean_p,actual_up_pct
bucket,,,
0,19451,0.354,44.769
1,19450,0.401,44.889
2,19451,0.425,46.440
3,19450,0.443,47.532
4,19451,0.458,47.792
5,19450,0.472,48.781
6,19450,0.485,49.321
7,19451,0.499,49.252
8,19450,0.516,50.370


in] test: model probability (10 buckets, low -> high) vs. actual share 'up first'


,windows,mean_p,actual_up_pct
bucket,,,
0,19246,0.257,43.703
1,19246,0.327,46.280
2,19246,0.366,45.843
3,19246,0.396,46.015
4,19246,0.422,46.378
5,19245,0.446,46.168
6,19246,0.469,47.568
7,19246,0.494,48.129
8,19246,0.528,48.909


AUC 0.50 = no direction information | 0.52–0.55 = weak | > 0.55 = interesting


,Setup,Train windows,Up first val %,Up first test %,AUC direction validation,AUC direction test,Fit seconds
0,15min,1080473,49.9036,47.8486,0.5226,0.5200,11.9047
1,60min,1064356,50.1908,46.7830,0.5152,0.5165,11.7844


## 7. Trading Simulation and Threshold Selection (validation only)

One position at a time. At every candle: `P_long`, `P_short`; if the larger one is ≥ threshold, a trade in that direction is
opened at the close; its result and duration come from the barrier outcome of that candle. Net = gross − costs.
The threshold with the best **t-value** (Ø net ÷ std × √trades) and at least `MIN_VALIDATION_TRADES` trades is selected.

In [10]:
def simulate(outcome: dict, rows: np.ndarray, direction: np.ndarray) -> pd.DataFrame:
    """rows: candidate candles in time order, direction: +1 / -1 per candidate."""
    trades, next_free = [], -1
    for r, d in zip(rows, direction):
        if r < next_free:
            continue
        side = outcome["long"] if d == 1 else outcome["short"]
        trades.append({"t": int(r), "direction": int(d), "gross_ticks": float(side["result"][r]),
                       "exit": int(side["exit"][r]), "minutes": int(side["minutes"][r])})
        next_free = r + int(side["minutes"][r])
    return pd.DataFrame(trades)


def summarize(trades: pd.DataFrame, timestamps: pd.Series, days: int) -> dict:
    """timestamps: all candles of the data set (trade rows index into it), days: trading days of the period."""
    if trades.empty:
        return {"Trades": 0}
    net = trades["gross_ticks"].to_numpy() - COST_PER_TRADE_TICKS
    equity = np.concatenate([[0.0], np.cumsum(net * TICK_VALUE_USD)])
    drawdown = (np.maximum.accumulate(equity) - equity).max()
    wins, losses = net[net > 0].sum(), -net[net < 0].sum()
    ts = timestamps.iloc[trades["t"].to_numpy()]
    monthly = pd.Series(net, index=ts.dt.strftime("%Y-%m").to_numpy()).groupby(level=0).sum()
    days = max(days, 1)
    is_long = trades["direction"].to_numpy() == 1
    return {
        "Trades": len(trades), "Trades/day": len(trades) / days, "Long %": 100 * is_long.mean(),
        "Target %": 100 * (trades["exit"] == EXIT_TARGET).mean(),
        "Stop %": 100 * (trades["exit"] == EXIT_STOP).mean(),
        "Time exit %": 100 * (trades["exit"] == EXIT_TIME).mean(),
        "Ø net (ticks)": net.mean(),
        "Ø net long": net[is_long].mean() if is_long.any() else np.nan,
        "Ø net short": net[~is_long].mean() if (~is_long).any() else np.nan,
        "Profit factor": wins / losses if losses > 0 else np.inf,
        "t-value": net.mean() / net.std(ddof=1) * np.sqrt(len(net)) if len(net) > 1 and net.std(ddof=1) > 0 else np.nan,
        "Months in profit": f"{int((monthly > 0).sum())}/{len(monthly)}",
        "Net total USD/contract": net.sum() * TICK_VALUE_USD,
        "Max drawdown USD": drawdown,
    }


def signals(p_long: np.ndarray, p_short: np.ndarray, threshold: float):
    best = np.maximum(p_long, p_short)
    mask = best >= threshold
    direction = np.where(p_long >= p_short, 1, -1)
    return mask, direction


probs = {"val": {}, "test": {}}
chosen = {}
for s in SETUPS:
    name = s["name"]
    o_tr = outcomes_train[name]
    va = val_rows[o_tr["valid"][val_rows]]
    p_l = models[name]["long"].predict_proba(X_train_all.iloc[va])[:, 1]
    p_s = models[name]["short"].predict_proba(X_train_all.iloc[va])[:, 1]
    probs["val"][name] = (va, p_l, p_s)

    val_days = df_train["timestamp"].iloc[va].dt.date.nunique()
    rows = []
    for thr in PROB_THRESHOLDS:
        mask, direction = signals(p_l, p_s, thr)
        tr = simulate(o_tr, va[mask], direction[mask])
        rows.append({"Threshold": thr, "Signal candles": int(mask.sum()),
                     **summarize(tr, df_train["timestamp"], val_days)})
    val_table = pd.DataFrame(rows)
    eligible = val_table[val_table["Trades"] >= MIN_VALIDATION_TRADES]
    if eligible.empty or eligible["t-value"].isna().all():
        chosen[name] = None
        print(f"[{name}] no threshold with ≥ {MIN_VALIDATION_TRADES} validation trades – setup not tradable")
    else:
        best = eligible.loc[eligible["t-value"].idxmax()]
        chosen[name] = float(best["Threshold"])
        print(f"[{name}] chosen threshold {chosen[name]:.3f}: validation Ø net {best['Ø net (ticks)']:+.2f} ticks, "
              f"t-value {best['t-value']:.2f}, {int(best['Trades'])} trades")
    print(f"[{name}] validation results per threshold:")
    display(val_table.round(3))

in] chosen threshold 0.500: validation Ø net -0.67 ticks, t-value -0.28, 131 trades
in] validation results per threshold:


,Threshold,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,150413,25164,58.521,39.358,36.497,36.477,27.027,-1.811,-1.514,-2.003,0.848,-10.626,1/17,-455620.0,464420.0
1,0.325,100810,19164,44.567,39.866,37.847,38.353,23.800,-2.036,-1.836,-2.169,0.835,-10.301,1/17,-390200.0,397310.0
2,0.350,63040,13517,31.435,41.052,40.164,39.706,20.130,-1.802,-1.735,-1.850,0.855,-7.603,2/17,-243640.0,249270.0
3,0.375,36682,8832,20.540,42.482,41.225,41.644,17.131,-1.762,-1.630,-1.860,0.861,-5.951,1/17,-155640.0,164900.0
4,0.400,19148,5340,12.419,45.337,41.985,43.258,14.757,-1.955,-1.959,-1.952,0.847,-5.196,1/17,-104400.0,116190.0
5,0.425,8396,2788,6.484,48.350,42.791,44.189,13.020,-1.857,-1.389,-2.295,0.856,-3.534,2/17,-51770.0,59420.0
6,0.450,2986,1194,2.777,52.513,45.059,45.142,9.799,-1.617,-1.115,-2.173,0.873,-2.000,4/17,-19310.0,25510.0
7,0.475,876,429,0.998,58.042,45.455,44.988,9.557,-0.713,0.896,-2.939,0.943,-0.528,8/17,-3060.0,9460.0
8,0.500,242,131,0.305,68.702,49.618,42.748,7.634,-0.672,0.278,-2.756,0.947,-0.276,7/17,-880.0,4500.0
9,0.525,58,40,0.093,75.000,55.000,42.500,2.500,2.900,3.633,0.700,1.240,0.623,9/14,1160.0,1940.0


in] chosen threshold 0.475: validation Ø net +0.66 ticks, t-value 0.37, 767 trades
in] validation results per threshold:


,Threshold,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,186344,7997,18.598,35.426,36.401,35.738,27.860,-1.805,-0.683,-2.420,0.918,-3.108,3/17,-144350.0,170690.0
1,0.325,146459,6800,15.814,36.426,37.397,37.912,24.691,-2.394,-1.600,-2.848,0.892,-3.882,2/17,-162780.0,182940.0
2,0.350,108566,5621,13.072,37.395,38.623,39.317,22.060,-2.294,-2.236,-2.328,0.895,-3.440,3/17,-128920.0,144940.0
3,0.375,75003,4382,10.191,37.038,39.799,40.507,19.694,-2.280,-2.335,-2.248,0.895,-3.088,3/17,-99900.0,117230.0
4,0.400,47750,3181,7.398,36.655,41.559,41.182,17.259,-1.334,-0.995,-1.530,0.935,-1.587,3/17,-42430.0,59210.0
5,0.425,28345,2175,5.058,37.425,42.437,42.667,14.897,-2.503,-0.982,-3.414,0.880,-2.534,5/17,-54450.0,62460.0
6,0.450,14879,1385,3.221,36.029,43.899,41.733,14.368,-0.864,0.050,-1.378,0.957,-0.684,7/17,-11960.0,32160.0
7,0.475,6853,767,1.784,36.636,45.502,39.505,14.993,0.661,1.683,0.070,1.032,0.372,8/17,5070.0,17810.0
8,0.500,3124,359,0.835,33.983,42.618,41.226,16.156,-2.994,1.516,-5.316,0.874,-1.090,6/17,-10750.0,16010.0
9,0.525,1537,196,0.456,31.633,38.265,43.878,17.857,-7.362,2.016,-11.701,0.730,-1.848,5/17,-14430.0,17840.0


## 8. Test: Model vs. Random Direction vs. Always Long

Applied **once** with the threshold chosen on validation. The comparisons use exactly the same entry candles.

In [11]:
test_days = df_test["timestamp"].dt.date.nunique()
test_rows = []
test_trades = {}
for s in SETUPS:
    name = s["name"]
    o_te = outcomes_test[name]
    te = np.flatnonzero(o_te["valid"])
    p_l = models[name]["long"].predict_proba(X_test.iloc[te])[:, 1]
    p_s = models[name]["short"].predict_proba(X_test.iloc[te])[:, 1]
    probs["test"][name] = (te, p_l, p_s)
    thr = chosen[name]
    if thr is None:
        continue
    mask, direction = signals(p_l, p_s, thr)
    rows_sig = te[mask]
    strategies = {
        "Model": [direction[mask]],
        f"Random direction (Ø {BASELINE_RANDOM_RUNS})": [
            np.random.default_rng(seed).choice([-1, 1], size=mask.sum()) for seed in range(BASELINE_RANDOM_RUNS)
        ],
        "Always long": [np.ones(mask.sum(), dtype=int)],
    }
    for strategy, dirs in strategies.items():
        sums = []
        for d in dirs:
            tr = simulate(o_te, rows_sig, d)
            sums.append(summarize(tr, df_test["timestamp"], test_days))
            if strategy == "Model":
                test_trades[name] = tr
        row = {"Setup": name, "Threshold": thr, "Strategy": strategy, "Signal candles": int(mask.sum())}
        if len(sums) == 1:
            row.update(sums[0])
        else:
            for key in sums[0]:
                vals = [x.get(key) for x in sums]
                row[key] = vals[0] if isinstance(vals[0], str) else float(np.nanmean(vals))
        test_rows.append(row)

test_df = pd.DataFrame(test_rows)
print(f"Rules: costs {COST_PER_TRADE_TICKS} ticks, target fill +{TARGET_FILL_EXTRA_TICKS} tick, stop first in the same "
      f"candle, no trades across pauses > {MAX_GAP_MINUTES} min, tick value {TICK_VALUE_USD} USD")
display(test_df.round(2))

Rules: costs 2.0 ticks, target fill +1 tick, stop first in the same candle, no trades across pauses > 5 min, tick value 10.0 USD


,Setup,Threshold,Strategy,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,15min,0.50,Model,321,141.0,0.45,58.16,40.43,43.26,16.31,-5.26,-4.88,-5.78,0.77,-1.40,4/12,-7410.0,11040.0
1,15min,0.50,Random direction (Ø 5),321,141.2,0.45,55.80,40.09,43.34,16.57,-3.96,-3.65,-4.42,0.84,-1.06,8/12,-5604.0,9786.0
2,15min,0.50,Always long,321,141.0,0.45,100.00,39.01,44.68,16.31,-2.09,-2.09,NaN,0.90,-0.56,6/12,-2950.0,7360.0
3,60min,0.48,Model,7293,681.0,2.18,18.36,40.53,38.47,21.00,1.74,-0.80,2.31,1.04,0.39,7/13,11840.0,29360.0
4,60min,0.48,Random direction (Ø 5),7293,680.8,2.18,49.56,39.16,39.80,21.03,-1.49,-5.14,2.10,0.98,-0.33,9/13,-10184.0,43470.0
5,60min,0.48,Always long,7293,680.0,2.17,100.00,37.50,41.62,20.88,-5.47,-5.47,NaN,0.90,-1.22,4/13,-37180.0,40720.0


### 8.1 For Information Only: All Thresholds on the Test Data

Do **not** pick the threshold from this table – that would be fitting to the test year. It only shows whether the
result is stable around the chosen threshold.

In [12]:
for s in SETUPS:
    name = s["name"]
    te, p_l, p_s = probs["test"][name]
    rows = []
    for thr in PROB_THRESHOLDS:
        mask, direction = signals(p_l, p_s, thr)
        tr = simulate(outcomes_test[name], te[mask], direction[mask])
        rows.append({"Threshold": thr, **summarize(tr, df_test["timestamp"], test_days)})
    print(f"[{name}] test results per threshold (chosen on validation: {chosen[name]})")
    display(pd.DataFrame(rows).round(3))

in] test results per threshold (chosen on validation: 0.5)


,Threshold,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,17789,56.834,29.636,35.258,35.275,29.468,-2.524,-1.967,-2.759,0.905,-5.054,0/13,-449030.0,453370.0
1,0.325,13574,43.367,30.116,36.327,36.857,26.816,-2.950,-1.874,-3.413,0.889,-5.290,0/13,-400370.0,404660.0
2,0.350,9502,30.358,31.751,37.929,37.918,24.153,-2.654,-2.045,-2.938,0.899,-4.169,1/13,-252210.0,253940.0
3,0.375,6069,19.390,34.932,40.040,38.771,21.190,-1.943,-0.772,-2.572,0.923,-2.560,4/13,-117920.0,123470.0
4,0.400,3472,11.093,39.113,41.043,39.516,19.441,-1.818,-1.484,-2.033,0.926,-1.903,3/13,-63120.0,67820.0
5,0.425,1803,5.760,43.206,43.039,40.488,16.473,-1.016,-0.039,-1.760,0.957,-0.793,5/13,-18320.0,32780.0
6,0.450,849,2.712,46.996,44.523,40.518,14.959,1.000,0.226,1.687,1.045,0.557,6/13,8490.0,15180.0
7,0.475,345,1.102,51.014,42.899,43.478,13.623,-3.649,-2.324,-5.030,0.847,-1.348,6/13,-12590.0,19050.0
8,0.500,141,0.450,58.156,40.426,43.262,16.312,-5.255,-4.878,-5.780,0.766,-1.402,4/12,-7410.0,11040.0
9,0.525,59,0.188,61.017,45.763,44.068,10.169,-1.864,-2.889,-0.261,0.910,-0.332,4/10,-1100.0,3360.0


in] test results per threshold (chosen on validation: 0.475)


,Threshold,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.300,5685,18.163,23.958,34.653,34.371,30.976,-1.086,-4.428,-0.033,0.978,-0.673,6/13,-61740.0,106890.0
1,0.325,4789,15.300,23.408,36.333,34.746,28.920,0.062,-1.037,0.399,1.001,0.036,4/13,2990.0,75260.0
2,0.350,3962,12.658,23.599,36.396,35.866,27.739,-0.746,-1.619,-0.476,0.984,-0.404,6/13,-29540.0,89030.0
3,0.375,3090,9.872,23.592,37.638,37.508,24.854,-1.621,-0.560,-1.949,0.966,-0.788,4/13,-50100.0,82230.0
4,0.400,2280,7.284,22.807,37.895,37.588,24.518,-1.666,1.181,-2.507,0.965,-0.713,6/13,-37980.0,65920.0
5,0.425,1606,5.131,22.167,37.360,39.913,22.727,-5.022,-5.025,-5.022,0.898,-1.813,4/13,-80660.0,94150.0
6,0.450,1068,3.412,21.161,38.670,39.513,21.816,-4.958,-9.270,-3.800,0.901,-1.436,6/13,-52950.0,83050.0
7,0.475,681,2.176,18.355,40.529,38.473,20.999,1.739,-0.800,2.309,1.036,0.388,7/13,11840.0,29360.0
8,0.500,454,1.450,12.996,39.427,41.630,18.943,-3.205,-14.746,-1.481,0.940,-0.556,5/13,-14550.0,33280.0
9,0.525,294,0.939,7.823,35.714,45.238,19.048,-11.799,-32.087,-10.077,0.807,-1.534,4/13,-34690.0,48930.0


## 8.2 Direction Strategy: Direction Model + Timing

* **Direction:** short if `P(up first)` is in the lowest `X %`, long if it is in the highest `X %` (cut-offs from the
  validation distribution, `DIRECTION_QUANTILES`).
* **Timing:** additionally the win model of the traded side must be at least at its validation quantile
  `TIMING_QUANTILES` (0 = no filter) – i.e. a move large enough to reach the target is expected.
* **Sides:** both, short only, long only (`DIRECTION_SIDES`).
* Same exits as above (volatility target / stop, stop first in the same candle, +1 tick for the target, time exit),
  one position at a time, costs included.

All combinations are evaluated on the **validation period**; the best t-value (with at least `MIN_VALIDATION_TRADES`
trades) is applied **once** to the test data and compared with random direction, always long and always short on exactly
the same entry candles. Long and short are reported separately.

In [13]:
def direction_signals(p_up, p_long, p_short, cfg):
    long_ = (p_up >= cfg["p_up_high"]) & (p_long >= cfg["p_long_min"])
    short_ = (p_up <= cfg["p_up_low"]) & (p_short >= cfg["p_short_min"])
    if cfg["side"] == "short":
        long_[:] = False
    elif cfg["side"] == "long":
        short_[:] = False
    return long_ | short_, np.where(long_, 1, -1)


dir_chosen, dir_test_rows, dir_trades, dir_signal_rows = {}, [], {}, {}
for s in SETUPS:
    name = s["name"]
    o_tr, o_te = outcomes_train[name], outcomes_test[name]
    va = val_rows[o_tr["valid"][val_rows]]
    te = np.flatnonzero(o_te["valid"])
    pu_va = direction_models[name].predict_proba(X_train_all.iloc[va])[:, 1]
    pu_te = direction_models[name].predict_proba(X_test.iloc[te])[:, 1]
    pl_va, ps_va = probs["val"][name][1], probs["val"][name][2]
    pl_te, ps_te = probs["test"][name][1], probs["test"][name][2]
    val_days = df_train["timestamp"].iloc[va].dt.date.nunique()

    # ---- grid on validation
    grid = []
    for q in DIRECTION_QUANTILES:
        for tq in TIMING_QUANTILES:
            for side in DIRECTION_SIDES:
                cfg = {
                    "direction_quantile": q, "timing_quantile": tq, "side": side,
                    "p_up_low": float(np.quantile(pu_va, q)), "p_up_high": float(np.quantile(pu_va, 1 - q)),
                    "p_long_min": float(np.quantile(pl_va, tq)) if tq > 0 else -np.inf,
                    "p_short_min": float(np.quantile(ps_va, tq)) if tq > 0 else -np.inf,
                }
                mask, d = direction_signals(pu_va, pl_va, ps_va, cfg)
                tr = simulate(o_tr, va[mask], d[mask])
                grid.append({**cfg, **summarize(tr, df_train["timestamp"], val_days)})
    grid_df = pd.DataFrame(grid)
    eligible = grid_df[(grid_df["Trades"] >= MIN_VALIDATION_TRADES) & grid_df["t-value"].notna()]
    print(f"[{name}] validation: all combinations")
    display(grid_df.drop(columns=["p_long_min", "p_short_min"]).round(3))
    if eligible.empty:
        print(f"[{name}] no combination with ≥ {MIN_VALIDATION_TRADES} validation trades")
        continue
    best = eligible.loc[eligible["t-value"].idxmax()]
    cfg = {k: best[k] for k in ["direction_quantile", "timing_quantile", "side", "p_up_low", "p_up_high",
                               "p_long_min", "p_short_min"]}
    dir_chosen[name] = cfg
    print(f"[{name}] chosen on validation: side {cfg['side']}, direction {cfg['direction_quantile']:.0%}, "
          f"timing quantile {cfg['timing_quantile']} -> validation Ø net {best['Ø net (ticks)']:+.2f} ticks, "
          f"t-value {best['t-value']:.2f}, {int(best['Trades'])} trades")

    # ---- test, applied once
    mask, d = direction_signals(pu_te, pl_te, ps_te, cfg)
    rows_sig = te[mask]
    dir_signal_rows[name] = (rows_sig, d[mask])
    n_sig = int(mask.sum())
    strategies = {
        "Direction model": [d[mask]],
        f"Random direction (Ø {BASELINE_RANDOM_RUNS})": [
            np.random.default_rng(seed).choice([-1, 1], size=n_sig) for seed in range(BASELINE_RANDOM_RUNS)
        ],
        "Always long": [np.ones(n_sig, dtype=int)],
        "Always short": [-np.ones(n_sig, dtype=int)],
    }
    for strategy, dirs in strategies.items():
        sums = []
        for dd in dirs:
            tr = simulate(o_te, rows_sig, dd)
            sums.append(summarize(tr, df_test["timestamp"], test_days))
            if strategy == "Direction model":
                dir_trades[name] = tr
        row = {"Setup": name, "Side": cfg["side"], "Direction quantile": cfg["direction_quantile"],
               "Timing quantile": cfg["timing_quantile"], "Strategy": strategy, "Signal candles": n_sig}
        if len(sums) == 1:
            row.update(sums[0])
        else:
            for key in sums[0]:
                vals = [x.get(key) for x in sums]
                row[key] = vals[0] if isinstance(vals[0], str) else float(np.nanmean(vals))
        dir_test_rows.append(row)

    # ---- for information only: all combinations on the test data
    info = []
    for _, g in grid_df.iterrows():
        c = {k: g[k] for k in ["direction_quantile", "timing_quantile", "side", "p_up_low", "p_up_high",
                               "p_long_min", "p_short_min"]}
        m2, d2 = direction_signals(pu_te, pl_te, ps_te, c)
        info.append({"direction_quantile": c["direction_quantile"], "timing_quantile": c["timing_quantile"],
                     "side": c["side"], **summarize(simulate(o_te, te[m2], d2[m2]), df_test["timestamp"], test_days)})
    dir_signal_rows[name + "_info"] = pd.DataFrame(info)

print("TEST – direction strategy with the combination chosen on validation:")
dir_test_df = pd.DataFrame(dir_test_rows)
display(dir_test_df.round(2))
for s in SETUPS:
    if s["name"] + "_info" in dir_signal_rows:
        print(f"[{s['name']}] for information only – all combinations on the test data (do not pick from here):")
        display(dir_signal_rows[s["name"] + "_info"].round(3))

in] validation: all combinations


,direction_quantile,timing_quantile,side,p_up_low,p_up_high,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,0.417,0.550,10061,23.398,55.054,29.480,26.667,43.852,-0.907,-0.503,-1.401,0.931,-2.634,2/17,-91210.0,105810.0
1,0.05,0.00,short,0.417,0.550,4596,10.688,0.000,28.372,25.305,46.323,-1.327,NaN,-1.327,0.924,-2.061,5/17,-61000.0,77030.0
2,0.05,0.00,long,0.417,0.550,5598,13.019,100.000,30.511,27.688,41.801,-0.446,-0.446,NaN,0.954,-1.360,4/17,-24960.0,35380.0
3,0.05,0.50,both,0.417,0.550,7170,16.674,59.135,33.445,30.335,36.220,-0.941,-0.960,-0.914,0.932,-2.290,2/17,-67460.0,80280.0
4,0.05,0.50,short,0.417,0.550,2949,6.858,0.000,33.537,29.332,37.131,-0.899,NaN,-0.899,0.953,-1.069,7/17,-26520.0,42900.0
5,0.05,0.50,long,0.417,0.550,4248,9.879,100.000,33.404,31.026,35.570,-0.954,-0.954,NaN,0.908,-2.523,3/17,-40520.0,49910.0
6,0.05,0.75,both,0.417,0.550,4941,11.491,62.214,36.187,33.333,30.480,-0.902,-1.306,-0.238,0.935,-1.855,3/17,-44580.0,52060.0
7,0.05,0.75,short,0.417,0.550,1874,4.358,0.000,36.926,32.177,30.896,-0.283,NaN,-0.283,0.985,-0.268,10/17,-5310.0,24470.0
8,0.05,0.75,long,0.417,0.550,3077,7.156,100.000,35.749,34.059,30.192,-1.307,-1.307,NaN,0.879,-2.955,4/17,-40220.0,47540.0
9,0.10,0.00,both,0.439,0.534,18926,44.014,54.327,29.948,28.205,41.847,-1.273,-0.926,-1.685,0.897,-5.637,1/17,-240870.0,256170.0


in] chosen on validation: side short, direction 5%, timing quantile 0.75 -> validation Ø net -0.28 ticks, t-value -0.27, 1874 trades
in] validation: all combinations


,direction_quantile,timing_quantile,side,p_up_low,p_up_high,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,0.343,0.579,3215,7.477,57.854,28.647,28.180,43.173,-2.625,-0.221,-5.925,0.905,-2.173,4/17,-84390.0,96320.0
1,0.05,0.00,short,0.343,0.579,1410,3.279,0.000,27.163,27.092,45.745,-5.094,NaN,-5.094,0.867,-2.172,5/17,-71830.0,86270.0
2,0.05,0.00,long,0.343,0.579,1892,4.400,100.000,29.915,28.911,41.173,-0.170,-0.170,NaN,0.991,-0.150,10/17,-3220.0,34200.0
3,0.05,0.50,both,0.343,0.579,2403,5.588,60.300,31.752,30.628,37.620,-1.630,-0.604,-3.188,0.940,-1.198,5/17,-39160.0,60590.0
4,0.05,0.50,short,0.343,0.579,974,2.265,0.000,30.493,30.390,39.117,-2.744,NaN,-2.744,0.927,-0.978,5/17,-26730.0,56410.0
5,0.05,0.50,long,0.343,0.579,1462,3.400,100.000,32.627,30.780,36.594,-0.873,-0.873,NaN,0.957,-0.690,7/17,-12760.0,21260.0
6,0.05,0.75,both,0.343,0.579,1481,3.444,64.146,33.964,34.774,31.263,-2.494,-0.917,-5.316,0.907,-1.516,5/17,-36940.0,49490.0
7,0.05,0.75,short,0.343,0.579,537,1.249,0.000,31.285,35.382,33.333,-5.151,NaN,-5.151,0.865,-1.415,8/17,-27660.0,44240.0
8,0.05,0.75,long,0.343,0.579,954,2.219,100.000,35.639,34.486,29.874,-0.958,-0.958,NaN,0.954,-0.621,6/17,-9140.0,18100.0
9,0.10,0.00,both,0.383,0.551,5770,13.419,57.539,29.965,28.406,41.629,-1.381,-0.039,-3.198,0.943,-1.716,5/17,-79660.0,107210.0


in] chosen on validation: side long, direction 10%, timing quantile 0.0 -> validation Ø net -0.06 ticks, t-value -0.08, 3446 trades
TEST – direction strategy with the combination chosen on validation:


,Setup,Side,Direction quantile,Timing quantile,Strategy,Signal candles,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,15min,short,0.05,0.75,Direction model,20021,5203.0,16.62,0.00,36.04,32.75,31.21,-0.84,NaN,-0.84,0.97,-0.78,7/13,-43720.0,88980.0
1,15min,short,0.05,0.75,Random direction (Ø 5),20021,5203.2,16.62,49.73,33.99,34.62,31.40,-1.92,-3.41,-0.44,0.94,-1.79,6/13,-99910.0,128374.0
2,15min,short,0.05,0.75,Always long,20021,5206.0,16.63,100.00,31.81,36.67,31.52,-3.83,-3.83,NaN,0.88,-3.57,2/13,-199320.0,208900.0
3,15min,short,0.05,0.75,Always short,20021,5203.0,16.62,0.00,36.04,32.75,31.21,-0.84,NaN,-0.84,0.97,-0.78,7/13,-43720.0,88980.0
4,60min,long,0.10,0.00,Direction model,32474,2221.0,7.10,100.00,30.30,30.30,39.40,-0.78,-0.78,NaN,0.98,-0.30,7/13,-17300.0,51600.0
5,60min,long,0.10,0.00,Random direction (Ø 5),32474,2222.0,7.10,49.60,30.52,30.34,39.14,-1.72,-0.33,-3.09,0.96,-0.67,6/13,-38288.0,100058.0
6,60min,long,0.10,0.00,Always long,32474,2221.0,7.10,100.00,30.30,30.30,39.40,-0.78,-0.78,NaN,0.98,-0.30,7/13,-17300.0,51600.0
7,60min,long,0.10,0.00,Always short,32474,2222.0,7.10,0.00,30.24,30.83,38.93,-3.49,NaN,-3.49,0.93,-1.36,3/13,-77450.0,93100.0


in] for information only – all combinations on the test data (do not pick from here):


,direction_quantile,timing_quantile,side,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,15312,48.920,22.492,29.709,26.959,43.332,-0.839,-2.921,-0.235,0.972,-1.307,3/13,-128520.0,184330.0
1,0.05,0.00,short,11965,38.227,0.000,29.837,25.775,44.388,-0.081,NaN,-0.081,0.997,-0.106,7/13,-9680.0,113200.0
2,0.05,0.00,long,3501,11.185,100.000,29.049,30.820,40.131,-2.904,-2.904,NaN,0.877,-2.656,4/13,-101660.0,106800.0
3,0.05,0.50,both,10792,34.479,23.277,32.885,30.282,36.833,-0.821,-1.607,-0.582,0.972,-1.091,5/13,-88570.0,148320.0
4,0.05,0.50,short,8323,26.591,0.000,33.125,29.328,37.547,-0.470,NaN,-0.470,0.985,-0.527,5/13,-39140.0,105020.0
5,0.05,0.50,long,2524,8.064,100.000,32.132,33.241,34.628,-1.694,-1.694,NaN,0.928,-1.287,4/13,-42760.0,57520.0
6,0.05,0.75,both,6951,22.208,25.349,35.376,33.808,30.816,-1.532,-3.510,-0.861,0.948,-1.699,3/13,-106520.0,142920.0
7,0.05,0.75,short,5203,16.623,0.000,36.037,32.750,31.213,-0.840,NaN,-0.840,0.973,-0.782,7/13,-43720.0,88980.0
8,0.05,0.75,long,1768,5.649,100.000,33.371,36.765,29.864,-3.574,-3.574,NaN,0.856,-2.230,2/13,-63190.0,65930.0
9,0.10,0.00,both,21792,69.623,26.303,29.561,27.813,42.626,-1.401,-2.142,-1.136,0.950,-2.774,2/13,-305290.0,339400.0


in] for information only – all combinations on the test data (do not pick from here):


,direction_quantile,timing_quantile,side,Trades,Trades/day,Long %,Target %,Stop %,Time exit %,Ø net (ticks),Ø net long,Ø net short,Profit factor,t-value,Months in profit,Net total USD/contract,Max drawdown USD
0,0.05,0.00,both,4774,15.252,28.865,30.938,28.152,40.909,1.597,2.438,1.256,1.029,0.726,7/13,76260.0,107430.0
1,0.05,0.00,short,3483,11.128,0.000,30.577,27.275,42.148,1.084,NaN,1.084,1.018,0.398,7/13,37760.0,106610.0
2,0.05,0.00,long,1455,4.649,100.000,30.928,30.241,38.832,2.409,2.409,NaN,1.054,0.729,7/13,35050.0,37560.0
3,0.05,0.50,both,3510,11.214,28.348,32.821,30.570,36.610,1.101,1.795,0.827,1.021,0.475,6/13,38660.0,66810.0
4,0.05,0.50,short,2543,8.125,0.000,32.717,29.768,37.515,1.091,NaN,1.091,1.020,0.385,7/13,27750.0,55110.0
5,0.05,0.50,long,1023,3.268,100.000,33.040,32.649,34.311,1.799,1.799,NaN,1.041,0.475,7/13,18400.0,40420.0
6,0.05,0.75,both,2281,7.288,27.050,33.538,33.406,33.056,-1.109,-0.442,-1.356,0.979,-0.401,7/13,-25300.0,74230.0
7,0.05,0.75,short,1678,5.361,0.000,33.552,32.837,33.611,-1.294,NaN,-1.294,0.976,-0.398,6/13,-21720.0,63090.0
8,0.05,0.75,long,630,2.013,100.000,33.175,34.603,32.222,-0.740,-0.740,NaN,0.984,-0.145,5/13,-4660.0,28080.0
9,0.10,0.00,both,6463,20.649,30.265,30.729,28.903,40.368,0.703,0.182,0.929,1.013,0.394,5/13,45430.0,118160.0


## 9. Write Test CSV with Signals, Save Models and Log

In [14]:
# Output = original test CSV (all columns as text, timestamp unchanged) + probabilities and signals per setup
test_text = pd.read_csv(TEST_CSV_PATH, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
add = pd.DataFrame({"_bar": df_test["Bar"].to_numpy("int64")})
for s in SETUPS:
    name = s["name"]
    te, p_l, p_s = probs["test"][name]
    col_l, col_s, col_sig = f"P_long_{name}", f"P_short_{name}", f"Signal_{name}"
    add[col_l] = np.nan
    add[col_s] = np.nan
    add.loc[te, col_l] = np.round(p_l, 4)
    add.loc[te, col_s] = np.round(p_s, 4)
    add[col_sig] = 0
    if chosen[name] is not None:
        mask, direction = signals(p_l, p_s, chosen[name])
        add.loc[te[mask], col_sig] = direction[mask]
    add[f"Target_ticks_{name}"] = outcomes_test[name]["target_ticks"]
    add[f"Stop_ticks_{name}"] = outcomes_test[name]["stop_ticks"]
    add[f"P_up_first_{name}"] = np.nan
    add.loc[te, f"P_up_first_{name}"] = np.round(direction_models[name].predict_proba(X_test.iloc[te])[:, 1], 4)
    add[f"Trade_{name}"] = 0                         # trades actually taken (one position at a time)
    if name in test_trades and not test_trades[name].empty:
        add.loc[test_trades[name]["t"].to_numpy(), f"Trade_{name}"] = test_trades[name]["direction"].to_numpy()
    add[f"DirSignal_{name}"] = 0                     # direction strategy (section 8.2): signal / trades taken
    add[f"DirTrade_{name}"] = 0
    if name in dir_signal_rows:
        rows_sig, dirs = dir_signal_rows[name]
        add.loc[rows_sig, f"DirSignal_{name}"] = dirs
    if name in dir_trades and not dir_trades[name].empty:
        add.loc[dir_trades[name]["t"].to_numpy(), f"DirTrade_{name}"] = dir_trades[name]["direction"].to_numpy()

test_text["_bar"] = pd.to_numeric(test_text["Bar"]).astype("int64")
out = test_text.merge(add, on="_bar", how="left", validate="one_to_one").drop(columns="_bar")
out.to_csv(TEST_OUTPUT_CSV_PATH, sep=";", index=False, encoding="utf-8-sig")
print(f"✓ Written: {TEST_OUTPUT_CSV_PATH} ({len(out):,} rows, new columns: {list(add.columns[1:])})")

run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_rows = []
for s in SETUPS:
    name = s["name"]
    path = MODEL_DIR / f"barrier_{SYMBOL}_{name}_{run_ts}.joblib"
    joblib.dump({
        "models": models[name], "direction_model": direction_models.get(name),
        "direction_strategy": dir_chosen.get(name),
        "features": FEATURE_COLUMNS, "setup": s, "threshold": chosen[name],
        "vol_lookback": VOL_LOOKBACK, "min_barrier_ticks": MIN_BARRIER_TICKS,
        "trained_until": str(df_train["timestamp"].iloc[val_start - 1]), "input_columns": INPUT_COLUMNS,
        "use_time_features": USE_TIME_FEATURES, "tick_size": TICK_SIZE,
        "rsi_periods": RSI_PERIODS, "twap_windows": TWAP_WINDOWS, "vwap_windows": VWAP_WINDOWS,
    }, path)
    joblib.dump(joblib.load(path), MODEL_DIR / f"barrier_{SYMBOL}_{name}_latest.joblib")
    print(f"✓ Saved: {path.resolve()}")
    model_row = test_df[(test_df["Setup"] == name) & (test_df["Strategy"] == "Model")]
    auc = pd.DataFrame(auc_rows)
    log_rows.append({
        "run_timestamp": run_ts, "setup": name, "horizon": s["horizon"], "mode": s.get("mode", "fixed"),
        "target": s.get("target_ticks", s.get("target_sigma")), "stop": s.get("stop_ticks", s.get("stop_sigma")),
        "threshold": chosen[name],
        "auc_direction_val": next((r["AUC direction validation"] for r in direction_rows if r["Setup"] == name), None),
        "auc_direction_test": next((r["AUC direction test"] for r in direction_rows if r["Setup"] == name), None),
        "auc_val_long": float(auc[(auc.Setup == name) & (auc.Side == "long")]["AUC validation"].iloc[0]),
        "auc_val_short": float(auc[(auc.Setup == name) & (auc.Side == "short")]["AUC validation"].iloc[0]),
        "auc_test_long": float(auc[(auc.Setup == name) & (auc.Side == "long")]["AUC test"].iloc[0]),
        "auc_test_short": float(auc[(auc.Setup == name) & (auc.Side == "short")]["AUC test"].iloc[0]),
        **({k: model_row.iloc[0][k] for k in ["Trades", "Ø net (ticks)", "Profit factor", "t-value",
                                             "Months in profit", "Net total USD/contract", "Max drawdown USD"]}
           if len(model_row) else {}),
        **({f"dir_{k}": dir_chosen[name][k] for k in ["side", "direction_quantile", "timing_quantile"]}
           if name in dir_chosen else {}),
        **({f"dir_{k}": v for k, v in dir_test_df[(dir_test_df["Setup"] == name)
                                                    & (dir_test_df["Strategy"] == "Direction model")].iloc[0][
               ["Trades", "Ø net (ticks)", "Profit factor", "t-value", "Months in profit", "Net total USD/contract"]].items()}
           if len(dir_test_df) and ((dir_test_df["Setup"] == name) & (dir_test_df["Strategy"] == "Direction model")).any() else {}),
        "costs_ticks": COST_PER_TRADE_TICKS, "train_row_stride": TRAIN_ROW_STRIDE, "model_path": str(path.resolve()),
    })
log_df = pd.DataFrame(log_rows)
if LOG_PATH.exists():
    log_df = pd.concat([pd.read_csv(LOG_PATH, sep=";"), log_df], ignore_index=True)
log_df.to_csv(LOG_PATH, sep=";", index=False)
print(f"✓ Logged to {LOG_PATH.resolve()}")

✓ Written: E:\Programierung\Daten\Test_GC_M1_20_10_2025_20_10_2026_barrier_signals.csv (354,649 rows, new columns: ['P_long_15min', 'P_short_15min', 'Signal_15min', 'Target_ticks_15min', 'Stop_ticks_15min', 'P_up_first_15min', 'Trade_15min', 'DirSignal_15min', 'DirTrade_15min', 'P_long_60min', 'P_short_60min', 'Signal_60min', 'Target_ticks_60min', 'Stop_ticks_60min', 'P_up_first_60min', 'Trade_60min', 'DirSignal_60min', 'DirTrade_60min'])
✓ Saved: E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier\barrier_GC_15min_20261026_200123.joblib
✓ Saved: E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier\barrier_GC_60min_20261026_200123.joblib
✓ Logged to E:\Programierung\Hitrevo\tradingbrains\trading_brains\trading_brains_main_tft\models_barrier\barrier_log.csv
